In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("./datasets/cardekho_imputated.csv")

In [3]:
df

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,19537,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,19540,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,19541,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [6]:
df.drop(columns=["Unnamed: 0"], inplace=True)

In [7]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [11]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

In [12]:
X

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5
...,...,...,...,...,...,...,...,...,...,...,...,...
15406,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5
15407,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7
15408,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5
15409,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7


In [13]:
y

0         120000
1         550000
2         215000
3         226000
4         570000
          ...   
15406     250000
15407     925000
15408     425000
15409    1225000
15410    1200000
Name: selling_price, Length: 15411, dtype: int64

In [14]:
X.dtypes

car_name                 str
brand                    str
model                    str
vehicle_age            int64
km_driven              int64
seller_type              str
fuel_type                str
transmission_type        str
mileage              float64
engine                 int64
max_power            float64
seats                  int64
dtype: object

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ohe = OneHotEncoder(drop="first")
str_cat = X.select_dtypes(include=["str"]).columns

scaler = StandardScaler()
num_cat = X.select_dtypes(exclude=["str"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("OneHotEncoder", ohe, str_cat),
        ("StandardScaler", scaler, num_cat)
    ],
    remainder="passthrough"
)

X = preprocessor.fit_transform(X)

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [19]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor()

gbr_params = {
    "loss": ['squared_error', 'absolute_error', 'huber', 'quantile'],
    "learning_rate": [1, 0.1, 0.01, 0.001, 0.0001],
    "n_estimators": [50, 100, 150, 200, 250, 300],
    "criterion": ["friedman_mse", "squared_error"],
    "min_samples_split": [2, 3, 4, 5, 10],
    "max_depth": [2, 3, 4, 5, 10]    
}

random = RandomizedSearchCV(
    estimator=gbr,
    param_distributions=gbr_params,
    n_iter=100,
    cv=3,
    verbose=2,
    n_jobs=-1
)

random.fit(X_train, y_train)

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END criterion=squared_error, learning_rate=0.001, loss=exponential, max_depth=2, min_samples_split=10, n_estimators=150; total time=   0.1s
[CV] END criterion=friedman_mse, learning_rate=0.1, loss=log_loss, max_depth=3, min_samples_split=5, n_estimators=250; total time=   0.0s
[CV] END criterion=squared_error, learning_rate=0.01, loss=exponential, max_depth=10, min_samples_split=5, n_estimators=50; total time=   0.0s
[CV] END criterion=squared_error, learning_rate=0.01, loss=exponential, max_depth=5, min_samples_split=2, n_estimators=300; total time=   0.0s
[CV] END criterion=squared_error, learning_rate=0.01, loss=log_loss, max_depth=10, min_samples_split=2, n_estimators=50; total time=   0.0s
[CV] END criterion=friedman_mse, learning_rate=1, loss=exponential, max_depth=4, min_samples_split=5, n_estimators=300; total time=   0.0s
[CV] END criterion=friedman_mse, learning_rate=0.01, loss=exponential, max_depth=5, min_s

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",GradientBoostingRegressor()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'criterion': ['friedman_mse', 'squared_error'], 'learning_rate': [1, 0.1, ...], 'loss': ['squared_error', 'absolute_error', ...], 'max_depth': [2, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",100
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``

In [21]:
from sklearn.metrics import r2_score

y_pred = random.predict(X_test)

print(r2_score(y_test, y_pred))

0.9224052380321257
[CV] END criterion=squared_error, learning_rate=0.01, loss=exponential, max_depth=4, min_samples_split=4, n_estimators=50; total time=   0.2s
[CV] END criterion=squared_error, learning_rate=0.0001, loss=log_loss, max_depth=3, min_samples_split=10, n_estimators=200; total time=   0.0s
[CV] END criterion=friedman_mse, learning_rate=0.001, loss=log_loss, max_depth=10, min_samples_split=4, n_estimators=200; total time=   0.0s
[CV] END criterion=squared_error, learning_rate=1, loss=exponential, max_depth=3, min_samples_split=4, n_estimators=200; total time=   0.0s
[CV] END criterion=friedman_mse, learning_rate=0.01, loss=log_loss, max_depth=10, min_samples_split=4, n_estimators=300; total time=   0.0s
[CV] END criterion=friedman_mse, learning_rate=0.01, loss=exponential, max_depth=2, min_samples_split=10, n_estimators=150; total time=   0.0s
[CV] END criterion=friedman_mse, learning_rate=0.001, loss=log_loss, max_depth=3, min_samples_split=10, n_estimators=150; total time